# Event Detection — Sensor Fault vs Environmental Event

An anomaly at a single station is an instrument problem. The same anomaly
occurring simultaneously across many stations is a real air quality event.

**Sensor faults are independent. Air masses are not.**

If eight instruments in different suburbs all spike within the same hour, the
instruments are not the explanation.

| Output | Grain |
|---|---|
| `workspace.aq_gold.hourly_network_state` | hour × parameter |
| `workspace.aq_gold.hourly_classification` | hour × parameter |
| `workspace.aq_gold.air_quality_event` | one row per episode |

Validated against the January 2020 Black Summer bushfire smoke. Nothing in the
detection logic references dates, fires or seasons — it is a genuine
out-of-sample check.

In [0]:
from pyspark.sql import functions as F

## 1. Hourly network state

For each hour and pollutant: how many stations were reporting, and how many of
them were anomalous?

`insufficient_history` rows are excluded — a station with no baseline cannot be
called anomalous or normal.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.hourly_network_state AS
SELECT
  obs_time,
  parameter_code,
  COUNT(DISTINCT site_id)                                        AS stations_reporting,
  COUNT(DISTINCT CASE WHEN anomaly_class IN ('unusual','extreme')
                      THEN site_id END)                          AS stations_anomalous,
  COUNT(DISTINCT CASE WHEN anomaly_class = 'extreme'
                      THEN site_id END)                          AS stations_extreme,
  ROUND(AVG(value_clean), 3)                                     AS network_mean,
  ROUND(MAX(value_clean), 3)                                     AS network_max,
  ROUND(AVG(z_score), 2)                                         AS network_mean_z
FROM workspace.aq_gold.observation_anomaly
WHERE anomaly_class <> 'insufficient_history'
GROUP BY obs_time, parameter_code
""")

spark.sql("""
SELECT parameter_code,
       count(*) AS hours,
       round(avg(stations_reporting), 1) AS avg_stations,
       max(stations_anomalous) AS max_concurrent_anomalies
FROM workspace.aq_gold.hourly_network_state
GROUP BY 1 ORDER BY 1
""").show()

+--------------+-----+------------+------------------------+
|parameter_code|hours|avg_stations|max_concurrent_anomalies|
+--------------+-----+------------+------------------------+
|           NO2|50856|        16.9|                      15|
|         OZONE|50858|        17.0|                      17|
|          PM10|52606|        17.1|                      17|
|         PM2.5|52606|        16.8|                      15|
+--------------+-----+------------+------------------------+



## 2. Classification

Thresholds are expressed as a **fraction** of stations reporting, not a fixed
count, because the number of active stations varies year to year. Four anomalous
stations out of six is a very different signal from four out of nineteen.

| Class | Condition | Interpretation |
|---|---|---|
| `insufficient_network` | Under 3 stations reporting | Cannot judge |
| `quiet` | No anomalies | Normal air |
| `isolated_fault` | Exactly 1 station | Instrument problem |
| `localised` | Under 25% of stations | A nearby source — roadworks, industry, hazard burn |
| `widespread` | 25–50% | Regional event |
| `network_wide` | Over 50% | Severe regional event |

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.hourly_classification AS
SELECT
  *,
  ROUND(stations_anomalous / NULLIF(stations_reporting, 0), 3) AS anomalous_fraction,
  CASE
    WHEN stations_reporting < 3                          THEN 'insufficient_network'
    WHEN stations_anomalous = 0                          THEN 'quiet'
    WHEN stations_anomalous = 1                          THEN 'isolated_fault'
    WHEN stations_anomalous / stations_reporting < 0.25  THEN 'localised'
    WHEN stations_anomalous / stations_reporting < 0.50  THEN 'widespread'
    ELSE 'network_wide'
  END AS event_class
FROM workspace.aq_gold.hourly_network_state
""")

spark.sql("""
SELECT event_class, count(*) AS hours,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
FROM workspace.aq_gold.hourly_classification
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------------+------+-----+
|         event_class| hours|  pct|
+--------------------+------+-----+
|               quiet|180918|87.43|
|      isolated_fault| 15747| 7.61|
|           localised|  7445| 3.60|
|          widespread|  1937| 0.94|
|        network_wide|   497| 0.24|
|insufficient_network|   382| 0.18|
+--------------------+------+-----+



## 3. Episodes

Consecutive event hours are grouped into single episodes using the same
gaps-and-islands pattern used for exceedance runs and flatline detection. A
three-day smoke event is one episode, not 72 separate hourly rows.

`HAVING COUNT(*) >= 3` requires an episode to last at least three hours, which
filters brief coincidences.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.air_quality_event AS
WITH flagged AS (
  SELECT *,
         CASE WHEN event_class IN ('widespread','network_wide') THEN 1 ELSE 0 END AS is_event
  FROM workspace.aq_gold.hourly_classification
),
grouped AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY parameter_code ORDER BY obs_time)
           - ROW_NUMBER() OVER (PARTITION BY parameter_code, is_event ORDER BY obs_time)
           AS episode_group
  FROM flagged
)
SELECT
  parameter_code,
  MIN(obs_time)                     AS event_start,
  MAX(obs_time)                     AS event_end,
  COUNT(*)                          AS duration_hours,
  MAX(stations_anomalous)           AS peak_stations_affected,
  MAX(stations_reporting)           AS network_size,
  ROUND(MAX(network_max), 2)        AS peak_concentration,
  ROUND(AVG(network_mean), 2)       AS mean_concentration,
  ROUND(MAX(anomalous_fraction), 3) AS peak_fraction
FROM grouped
WHERE is_event = 1
GROUP BY parameter_code, episode_group
HAVING COUNT(*) >= 3
""")

spark.sql("""
SELECT parameter_code,
       count(*) AS episodes,
       max(duration_hours) AS longest_hours,
       round(avg(duration_hours), 1) AS avg_hours
FROM workspace.aq_gold.air_quality_event
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------+--------+-------------+---------+
|parameter_code|episodes|longest_hours|avg_hours|
+--------------+--------+-------------+---------+
|         PM2.5|      99|           14|      5.3|
|          PM10|      79|           10|      4.3|
|         OZONE|      78|           10|      4.8|
|           NO2|      65|           12|      4.2|
+--------------+--------+-------------+---------+



## 4. Validation against January 2020

Sydney experienced severe bushfire smoke through December 2019 and January 2020.
January 2020 is the first month of this dataset.

If the detector works, the longest and most severe PM2.5 episodes in the entire
six-year dataset should fall in January 2020.

In [0]:
print("=== Top 15 longest PM2.5 episodes, all years ===")
spark.sql("""
SELECT event_start, event_end, duration_hours,
       peak_stations_affected, network_size, peak_concentration
FROM workspace.aq_gold.air_quality_event
WHERE parameter_code = 'PM2.5'
ORDER BY duration_hours DESC
LIMIT 15
""").show(truncate=False)

=== Top 15 longest PM2.5 episodes, all years ===
+-------------------+-------------------+--------------+----------------------+------------+------------------+
|event_start        |event_end          |duration_hours|peak_stations_affected|network_size|peak_concentration|
+-------------------+-------------------+--------------+----------------------+------------+------------------+
|2020-08-29 18:00:00|2020-08-30 07:00:00|14            |12                    |16          |145.81            |
|2021-10-09 19:00:00|2021-10-10 08:00:00|14            |8                     |18          |1084.77           |
|2020-04-12 21:00:00|2020-04-13 08:00:00|12            |11                    |16          |44.9              |
|2020-08-28 21:00:00|2020-08-29 06:00:00|10            |8                     |16          |63.38             |
|2024-08-03 21:00:00|2024-08-04 06:00:00|10            |10                    |18          |80.58             |
|2020-08-01 20:00:00|2020-08-02 05:00:00|10            

In [0]:
print("=== Event hours by month, PM2.5, top 12 ===")
spark.sql("""
SELECT DATE_TRUNC('MONTH', obs_time) AS month,
       SUM(CASE WHEN event_class IN ('widespread','network_wide') THEN 1 ELSE 0 END) AS event_hours,
       ROUND(AVG(network_mean), 1) AS mean_pm25
FROM workspace.aq_gold.hourly_classification
WHERE parameter_code = 'PM2.5'
GROUP BY 1
ORDER BY event_hours DESC
LIMIT 12
""").show()

=== Event hours by month, PM2.5, top 12 ===
+-------------------+-----------+---------+
|              month|event_hours|mean_pm25|
+-------------------+-----------+---------+
|2020-08-01 00:00:00|         40|      7.5|
|2020-01-01 00:00:00|         36|     22.5|
|2020-05-01 00:00:00|         31|      6.7|
|2022-07-01 00:00:00|         27|      5.1|
|2021-08-01 00:00:00|         26|      8.9|
|2025-04-01 00:00:00|         24|      6.9|
|2023-09-01 00:00:00|         21|     11.6|
|2021-06-01 00:00:00|         18|      7.6|
|2020-06-01 00:00:00|         18|      7.9|
|2024-09-01 00:00:00|         18|      6.6|
|2023-06-01 00:00:00|         17|      7.7|
|2025-12-01 00:00:00|         16|      8.0|
+-------------------+-----------+---------+



### How to read the result

| Outcome | Meaning | What to write |
|---|---|---|
| January 2020 tops both lists | Strongest possible result | "Independently identified the January 2020 bushfire smoke episode as the most severe PM2.5 event in the dataset, with no date or seasonal input" |
| High but not first | Works, with competitors. Investigate what beat it — it may be a real event too | Report both, name what you found |
| Absent | See the baseline adaptation test below | Report it honestly |

## 5. The baseline adaptation test

**This is the check that matters most, whichever way it goes.**

The rolling baseline is 7 days. If smoke persisted for weeks, the baseline
**adapts** to it — after a few days the elevated readings look normal relative to
recent history, and the detector goes quiet in the middle of the worst air.

| What you see | Meaning |
|---|---|
| `mean_pm25` high and `max_anomalous` high throughout | The detector holds up |
| `mean_pm25` high but `max_anomalous` falls to zero after a few days | Baseline adaptation, demonstrated |

If it is the second, **that is a finding, not a failure.** Being the person who
noticed and quantified a known weakness in their own method is worth more than a
detector that only appeared to work.

In [0]:
spark.sql("""
SELECT DATE_TRUNC('DAY', obs_time) AS day,
       ROUND(AVG(network_mean), 1) AS mean_pm25,
       MAX(stations_anomalous)     AS max_anomalous,
       MAX(stations_reporting)     AS reporting
FROM workspace.aq_gold.hourly_network_state
WHERE parameter_code = 'PM2.5'
  AND obs_time >= '2020-01-01' AND obs_time < '2020-02-01'
GROUP BY 1 ORDER BY 1
""").show(35, truncate=False)

+-------------------+---------+-------------+---------+
|day                |mean_pm25|max_anomalous|reporting|
+-------------------+---------+-------------+---------+
|2020-01-01 00:00:00|28.6     |4            |14       |
|2020-01-02 00:00:00|29.2     |3            |15       |
|2020-01-03 00:00:00|21.4     |1            |14       |
|2020-01-04 00:00:00|34.2     |12           |14       |
|2020-01-05 00:00:00|67.1     |8            |13       |
|2020-01-06 00:00:00|11.9     |0            |14       |
|2020-01-07 00:00:00|17.3     |0            |14       |
|2020-01-08 00:00:00|76.7     |9            |15       |
|2020-01-09 00:00:00|17.8     |0            |15       |
|2020-01-10 00:00:00|13.5     |0            |15       |
|2020-01-11 00:00:00|38.7     |3            |15       |
|2020-01-12 00:00:00|47.1     |0            |15       |
|2020-01-13 00:00:00|23.1     |0            |15       |
|2020-01-14 00:00:00|7.6      |0            |15       |
|2020-01-15 00:00:00|6.6      |0            |15 

## 6. Seasonal baseline — the alternative

Run this only if the previous cell showed baseline adaptation.

Instead of comparing against the last 7 days, compare each reading against the
**same calendar month in other years** at the same station. A multi-week smoke
event cannot hide inside a baseline built from different years.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.seasonal_anomaly AS
WITH monthly_norm AS (
  SELECT site_id, parameter_code, month(obs_date) AS mth,
         AVG(value_clean)    AS seasonal_mean,
         STDDEV(value_clean) AS seasonal_sd
  FROM workspace.aq_silver.fact_observation
  GROUP BY 1, 2, 3
)
SELECT
  f.site_id, f.parameter_code, f.obs_time, f.obs_date, f.value_clean,
  ROUND(n.seasonal_mean, 3) AS seasonal_mean,
  ROUND((f.value_clean - n.seasonal_mean) / NULLIF(n.seasonal_sd, 0), 2) AS seasonal_z
FROM workspace.aq_silver.fact_observation f
JOIN monthly_norm n
  ON f.site_id = n.site_id
 AND f.parameter_code = n.parameter_code
 AND month(f.obs_date) = n.mth
""")

spark.sql("""
SELECT DATE_TRUNC('MONTH', obs_time) AS month,
       ROUND(AVG(seasonal_z), 2) AS mean_seasonal_z,
       SUM(CASE WHEN seasonal_z > 3 THEN 1 ELSE 0 END) AS extreme_readings
FROM workspace.aq_gold.seasonal_anomaly
WHERE parameter_code = 'PM2.5'
GROUP BY 1
ORDER BY mean_seasonal_z DESC
LIMIT 10
""").show()

+-------------------+---------------+----------------+
|              month|mean_seasonal_z|extreme_readings|
+-------------------+---------------+----------------+
|2020-01-01 00:00:00|           1.08|            1083|
|2023-09-01 00:00:00|           0.47|             634|
|2021-04-01 00:00:00|           0.42|             368|
|2024-03-01 00:00:00|           0.37|             323|
|2021-08-01 00:00:00|           0.28|             479|
|2020-11-01 00:00:00|           0.27|             283|
|2025-12-01 00:00:00|           0.26|             336|
|2023-07-01 00:00:00|            0.2|             122|
|2020-07-01 00:00:00|            0.2|             300|
|2021-05-01 00:00:00|           0.19|             382|
+-------------------+---------------+----------------+



### Comparing the two baselines

If the rolling baseline missed January 2020 and the seasonal baseline caught it,
you have demonstrated something concrete: **the choice of baseline determines what
a detector can see.** A 7-day window is blind to events longer than a week.

That comparison is a stronger result than either detector alone.

## 7. The discrimination summary

The project's headline table. Of every anomaly in six years of Sydney air quality
data, how many were instruments failing and how many were the air genuinely being
bad — and here is how they were told apart.

In [0]:
spark.sql("""
SELECT
  event_class,
  count(*) AS hours,
  round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct,
  CASE event_class
    WHEN 'isolated_fault'       THEN 'Single station - instrument fault likely'
    WHEN 'localised'            THEN 'Few stations - local source'
    WHEN 'widespread'           THEN 'Many stations - regional event'
    WHEN 'network_wide'         THEN 'Majority of network - severe regional event'
    WHEN 'quiet'                THEN 'No anomalies'
    ELSE 'Too few stations reporting to judge'
  END AS interpretation
FROM workspace.aq_gold.hourly_classification
GROUP BY 1 ORDER BY 2 DESC
""").show(truncate=False)

+--------------------+------+-----+-------------------------------------------+
|event_class         |hours |pct  |interpretation                             |
+--------------------+------+-----+-------------------------------------------+
|quiet               |180918|87.43|No anomalies                               |
|isolated_fault      |15747 |7.61 |Single station - instrument fault likely   |
|localised           |7445  |3.60 |Few stations - local source                |
|widespread          |1937  |0.94 |Many stations - regional event             |
|network_wide        |497   |0.24 |Majority of network - severe regional event|
|insufficient_network|382   |0.18 |Too few stations reporting to judge        |
+--------------------+------+-----+-------------------------------------------+



## 8. Prove the grain

In [0]:
checks = [
    ("hourly_network_state",   "obs_time, parameter_code"),
    ("hourly_classification",  "obs_time, parameter_code"),
    ("air_quality_event",      "parameter_code, event_start"),
]

for table, keys in checks:
    n = spark.sql(f"""
        SELECT {keys}, count(*) AS c
        FROM workspace.aq_gold.{table}
        GROUP BY {keys} HAVING count(*) > 1
    """).count()
    print(f"{table:24} grain violations: {n}")

hourly_network_state     grain violations: 0
hourly_classification    grain violations: 0
air_quality_event        grain violations: 0


## 9. Table comments — feeds Genie

In [0]:
spark.sql("""
COMMENT ON TABLE workspace.aq_gold.hourly_classification IS
  'One row per hour, per pollutant. Classifies each hour by how many stations across the network were simultaneously anomalous, distinguishing isolated instrument faults from regional air quality events.'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.air_quality_event IS
  'One row per detected air quality episode: a run of three or more consecutive hours in which a quarter or more of reporting stations were anomalous. Includes duration, peak concentration and how much of the network was affected.'
""")

spark.sql("""
ALTER TABLE workspace.aq_gold.hourly_classification
  ALTER COLUMN event_class COMMENT 'isolated_fault means one station only and is likely an instrument problem. widespread and network_wide indicate a real environmental event, because sensor faults are independent while air masses are not'
""")

print("comments applied")

comments applied
